In [1]:
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama,OllamaEmbeddings
from langchain_community.vectorstores import FAISS



Indexing

In [ ]:

video_id="YIszsqhLGIs"

try:
    # Create API instance
    api = YouTubeTranscriptApi()
    # Fetch transcript
    transcript_list = api.fetch(video_id, languages=["hi"])

    # Flatten it to plain text
    # transcript = " ".join(chunk["text"] for chunk in transcript_list)
    transcript = " ".join(getattr(chunk, "text", "") for chunk in transcript_list)

    # print("Transcript fetched successfully ✅\n")
    # print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")


In [3]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text="You feel successful people don't gossip.", start=0.08, duration=3.28), FetchedTranscriptSnippet(text='>> Successful people actually are', start=1.68, duration=3.599), FetchedTranscriptSnippet(text='extraordinary at keeping secrets. They', start=3.36, duration=3.6), FetchedTranscriptSnippet(text='gossip to know information. You will', start=5.279, duration=3.841), FetchedTranscriptSnippet(text="never see a wealthy person who's", start=6.96, duration=4.08), FetchedTranscriptSnippet(text='terrible at keeping secrets. The moment', start=9.12, duration=3.599), FetchedTranscriptSnippet(text="second third generation who's not good", start=11.04, duration=3.36), FetchedTranscriptSnippet(text='at keeping secrets the wealth will go', start=12.719, duration=3.441), FetchedTranscriptSnippet(text='away. Were you good at keeping secrets', start=14.4, duration=3.44), FetchedTranscriptSnippet(text='from childhood? I was good at keeping', start

Indexing


In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [5]:
len(chunks)

173

In [6]:
chunks[12]

Document(metadata={}, page_content="society collective society simp >> right when we were you would remember that your grandparents would tell you that and we used to wear the same shirt right but now imagine trying to tell three cousins you're going to wear the same clothes it's not okay they have become individuals now right so as we move from let's say a non-indiv individualistic society to individualistic we will get to more failure understanding choices and so on and so forth but there's another thing that is happening which is preventing failure again >> is people fearing their social media followers >> right that let's say you are a young person and you have like managed to get 2,000 followers in your head you're already a celebrity Raj you will remember when you had 50,000 followers and you would get 40 likes, your reaction would be the same as today at millions of followers, right? The the threshold has changed. >> Yeah. >> Right. But unfortunately, we get shaped by our audien

In [7]:
#gneerate embedding and store in vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(chunks, embeddings)

In [8]:
vector_store.index_to_docstore_id

{0: '3613f4ab-9737-4721-9327-e1a7dd91016c',
 1: 'db591044-9746-4a40-b25a-70c5bfca92ff',
 2: '41fa7615-911a-4223-8a52-26d42f26c412',
 3: '7bc00772-9770-4571-a116-39f834c86b05',
 4: '6c04309c-2cab-4c61-923f-8d90347778d7',
 5: '04d78cad-7259-4488-962c-6cb936af819f',
 6: '373ae944-8a58-4779-8dcf-384678470003',
 7: '3048ccfe-cab2-4671-9e60-9081febe2e6f',
 8: '6d759f94-af64-44fa-a4bb-770f47eff0f2',
 9: 'addcf94a-7b00-4e68-bb98-2cb81cc66c63',
 10: 'c33c46c7-a2a7-4251-8893-1462aee543a6',
 11: 'd8d541e4-196b-4564-825b-3a9de1358b5b',
 12: 'efde2dbd-e830-41c3-afe5-c8e67125028d',
 13: 'e9ad8656-f7a7-42c4-a808-27d09ae9ca1f',
 14: 'ec8bd199-42c6-46f0-a7f6-c612cc360621',
 15: '09f4912b-cdc2-432b-886f-f4c045868491',
 16: '5f7e62a7-abb1-4600-b6fa-c235f3d02429',
 17: '377711e5-662c-4fa5-98f0-a9930c881cd7',
 18: 'cd36188d-bfe7-453c-882b-375e5a19d4e8',
 19: 'bba41b2b-770d-47aa-97c2-d823fa59fa74',
 20: 'd23b9518-6fda-4edf-8a78-6b32eb4ea45a',
 21: 'b200ba5f-6153-4c46-9bf1-7335437f532c',
 22: '6bb7101f-81bb-

In [9]:
vector_store.get_by_ids(['2436bdb8-3f5f-49c6-8915-0c654c888700'])

[]

Building a chain

In [10]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [11]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [12]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [13]:
# retriever--------                                     
    #Input → question (string)                                
    #Output → list of Document objects


#RunnableLambda(format_docs)--------->>We convert a list of Document objects into a string because LLMs only understand plain text, not Python objects.
   #Input → List[Document]     
   #Output → single formatted string



parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),   
    'question': RunnablePassthrough()   #passes the question unchanged.
})

In [14]:
parallel_chain.invoke('who is kunal')

{'context': "man. Good stuff. >> Thank you. Done. I'm actually happy. I was nervous. I'm a fan more than and more than the kunala. The person I'm fan of kunal sha the person who tweets born and brought up. I'm actually born uh in Ahmedabad. My mom's from here but I've always been here. So my I was like raised >> Bangalore. Bombay. >> No Bombay only. >> Okay. >> Bangalore is only less Indian. >> Yeah. It's just that I never got like good at it. >> Okay. >> Uh I can speak uh it's not that I can't speak Hindi but I know that I will struggle with words. >> You think in English? >> That's a good question. I don't know what language I think in >> but you read and consume everything in English. >> H no I I do consume in multiple languages. It's just that I don't know what language do I think in become a I think it's become a mixture now. Okay. >> Is it work? I don't know. I [\xa0__\xa0] maybe I don't I don't have conversations with myself when I'm thinking. Okay. >> I'm usually just as a trig

In [15]:
parser = StrOutputParser()

In [16]:
llm = ChatOllama(model="ministral-3:8b", temperature=0.2)

In [17]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [18]:
main_chain = parallel_chain | prompt | llm | parser

In [20]:
main_chain.invoke('Can you suggest any one title of this video')

'Based on the provided transcript context, here are a few possible title suggestions for the video:\n\n1. **"From Survival to Legacy: Lessons from Tata Group’s First Five Years"**\n2. **"The Unseen Side of Success: Entrepreneurship, Failure, and the Power of Stories"**\n3. **"Beyond the Hype: The Real Juice of Entrepreneurship and Leadership"**\n4. **"Why Failure Stories Matter: Redefining Success Through Entrepreneurial Journeys"**\n5. **"The Experiment of Building: Lessons from Tata Group’s Early Days"**'